# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第15章 不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `sim_f0_tracks.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」(第12章のGAMM想定例と対応)

目的:
  scripts/R/30_gamm_f0.R が読み込む data/processed/f0_tracks.csv を、
  東京方言アクセントの核あり語・無核語という想定例に沿ってシミュレートする。
  **実測データではない。** 核あり語はアクセント核の直後(正規化時間の
  中盤)で急激なF0下降(H*+L)を示すのに対し、無核語はなだらかに
  上昇を続けるパターンとして設計する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)
library(tidyverse)

n_speakers   <- 15
n_utt_per    <- 6   # 話者ごとの発話数(条件あたり3発話)
n_timepoints <- 21  # 正規化時間 0 から 1 まで

speaker_mean_f0 <- tibble(
  speaker = sprintf("sp%02d", 1:n_speakers),
  base_f0 = rnorm(n_speakers, 180, 25)
)

time_norm <- seq(0, 1, length.out = n_timepoints)

make_curve <- function(condition, base_f0) {
  if (condition == "核あり") {
    # 立ち上がり(~0.3)でピーク、その直後(H*+L)で急落
    20 * plogis((time_norm - 0.25) / 0.05) - 35 * plogis((time_norm - 0.5) / 0.04) + base_f0
  } else {
    # 無核: なだらかな上昇を保つ
    15 * plogis((time_norm - 0.4) / 0.15) + base_f0
  }
}

d <- speaker_mean_f0 |>
  tidyr::crossing(utt_in_speaker = 1:n_utt_per) |>
  mutate(condition = rep(c("核あり", "無核"), length.out = n()),
         utterance = paste0(speaker, "_u", utt_in_speaker)) |>
  rowwise() |>
  mutate(curve = list(make_curve(condition, base_f0) + rnorm(n_timepoints, 0, 4))) |>
  ungroup() |>
  select(speaker, utterance, condition, curve) |>
  mutate(time_norm = list(time_norm)) |>
  tidyr::unnest(cols = c(curve, time_norm)) |>
  rename(f0 = curve) |>
  select(speaker, utterance, condition, time_norm, f0)

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
write_csv(d, here::here("data", "processed", "f0_tracks.csv"))
cat("生成した行数:", nrow(d), " / 話者数:", n_speakers,
    " / 発話数:", n_distinct(d$utterance), "\n")


## `30_gamm_f0.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」（第12章の問題を受ける）

なぜ GAMM を使うのか:
  F0 曲線を「平均 F0」や「最大値」といった1点に要約すると、
  曲線の形そのものが持つ情報（いつ上がり、いつ下がるか）が失われる。
  韻律理論の予測は多くの場合その形についての予測であり、
  要約統計量では検証にならない。GAMM は時間の関数として F0 を推定し、
  条件間の差が時間軸のどこに生じるかを示せる。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(mgcv)
library(itsadug)

d <- read_csv(here::here("data", "processed", "f0_tracks.csv"),
              show_col_types = FALSE) |>
  mutate(
    speaker   = factor(speaker),
    condition = factor(condition),
    # 話者内で半音正規化する。Hz のままだと話者の声域差が条件効果を覆う。
    f0_st = 12 * log2(f0 / ave(f0, speaker, FUN = function(x) median(x, na.rm = TRUE)))
  ) |>
  arrange(speaker, utterance, time_norm) |>
  group_by(speaker, utterance) |>
  mutate(start_event = row_number() == 1) |>
  ungroup()

# 条件ごとの平滑 + 話者ごとのランダム平滑。
# 時系列の自己相関を無視すると有意差が出すぎるため、rho で補正する。
m0 <- bam(f0_st ~ condition +
            s(time_norm, by = condition, k = 20) +
            s(time_norm, speaker, bs = "fs", m = 1),
          data = d, method = "fREML", discrete = TRUE)

rho <- start_value_rho(m0)
m1 <- bam(f0_st ~ condition +
            s(time_norm, by = condition, k = 20) +
            s(time_norm, speaker, bs = "fs", m = 1),
          data = d, method = "fREML", discrete = TRUE,
          rho = rho, AR.start = d$start_event)

print(summary(m1))
gam.check(m1)   # k が足りているかを確認する

# 条件間の差が「時間軸のどこに」あるかを見る。
# これが GAMM を使う最大の理由である（第12章の問いへの答え）。
# 基本グラフィックスの既定フォントには日本語グリフが無く、軸ラベルが□に
# 化けるため、日本語フォントを明示する(Macはヒラギノ、それ以外はshowtextで
# 登録済みの"notosans")。
fig_path <- here::here("assets", "figures", "ch15", "fig15-02-f0-difference.png")
if (Sys.info()[["sysname"]] == "Darwin") {
  quartzFonts(japanese = quartzFont(rep("Hiragino Sans", 4)))
  png(fig_path, width = 1600, height = 1100, res = 320, type = "quartz", family = "japanese")
  par(family = "japanese")
} else {
  png(fig_path, width = 1600, height = 1100, res = 320, type = "cairo")
  par(family = "notosans")
}
# sim.ci = TRUE: 既定の逐点(pointwise)信頼区間ではなく、時間軸全体を通じた
# 同時(simultaneous)信頼区間を用いる。逐点区間で「有意な区間」を報告すると、
# 多数の時点それぞれで独立に検定したのと同じことになり、多重比較の問題で
# 有意な区間を過大に検出しやすい。
plot_diff(m1, view = "time_norm",
          comp = list(condition = levels(d$condition)[1:2]),
          sim.ci = TRUE,
          main = "", xlab = "正規化時間", ylab = "差 (半音)",
          hide.label = TRUE,   # 右端の英語自動ラベルは消す(区間の定義は本文・キャプションで述べる)
          col.diff = "grey15") # 有意区間の印は赤でなく濃い灰色(白黒印刷でも判別できる)
dev.off()


## `ch15_worked_example.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」通しの作例

「事前分布の選択と感度分析」節で述べた、架空の説明用の数値例
(事後平均0.42、95%信用区間[0.05, 0.79]相当)を、実際に正規分布から
20,000サンプルを描いてROPE([-0.1, 0.1])との重なりを図示する。
ここでの数値そのものは架空の説明用サンプルであり、実データの解析結果ではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)

# 事後分布の模式図(ベイズ推定の考え方を説明するための、架空の説明用サンプル)
posterior_draws <- rnorm(20000, mean = 0.42, sd = 0.19)
rope <- c(-0.1, 0.1)
pct_in_rope <- mean(posterior_draws > rope[1] & posterior_draws < rope[2]) * 100
ci95 <- quantile(posterior_draws, c(0.025, 0.975))

d <- tibble(beta = posterior_draws)
p <- ggplot(d, aes(beta)) +
  geom_density(fill = "#34518A", alpha = 0.25, colour = "#34518A", linewidth = 0.8) +
  geom_vline(xintercept = rope, linetype = 2, colour = "#A8681F") +
  geom_vline(xintercept = ci95, linetype = 3, colour = "grey30") +
  annotate("rect", xmin = rope[1], xmax = rope[2], ymin = 0, ymax = Inf,
           fill = "#A8681F", alpha = 0.12) +
  labs(x = expression(beta ~ "(効果量、架空の説明用サンプル)"), y = "事後密度",
       caption = paste0("破線・網掛けはROPE [-0.1, 0.1]。事後分布の", round(pct_in_rope, 1),
                         "%がROPE内。説明用の模式図であり実データではない。scripts/R/ch15_worked_example.R より生成。"))
save_fig(p, "fig15-01-posterior-rope", chapter = "ch15", width = 6, height = 3.6)
cat("ROPE内の事後確率:", round(pct_in_rope, 1), "%\n")


## `ch15_fda_registration.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」

「FDA——曲線を関数として扱う」節の作例。同じ形状(単一のピークを持つ
F0曲線)を持つが、ピークの生じるタイミングが話者ごとに異なる架空の
曲線群を作り、時間軸をそのまま重ねた場合(位置合わせ前)と、各曲線の
ピーク位置を共通の基準点に揃えてから重ねた場合(位置合わせ後、
landmark registration)とで、曲線群の平均がどう変わるかを比較する。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_curves <- 12
t_grid   <- seq(0, 1, length.out = 200)

peak_times <- runif(n_curves, 0.35, 0.65)   # 話者ごとにピークの時刻がばらつく

make_curve <- function(peak_t) {
  100 + 30 * exp(-((t_grid - peak_t) / 0.08)^2)
}

curves <- map_dfr(seq_len(n_curves), function(i) {
  tibble(curve_id = i, t = t_grid, f0 = make_curve(peak_times[i]) + rnorm(length(t_grid), 0, 1.5))
})

# --- 位置合わせ前の平均 ---------------------------------------------------
mean_unaligned <- curves |> group_by(t) |> summarise(f0 = mean(f0), .groups = "drop")

# --- landmark registrationによる位置合わせ ------------------------------
# 各曲線のピーク時刻を、全曲線の平均ピーク時刻に揃えるよう、時間軸を
# 区分線形に伸縮する(0とt=1は固定したまま、ピーク位置だけを動かす)。
target_peak <- mean(peak_times)
warp_time <- function(t, peak_t, target) {
  ifelse(t <= peak_t,
         t / peak_t * target,
         target + (t - peak_t) / (1 - peak_t) * (1 - target))
}

curves_aligned <- curves |>
  group_by(curve_id) |>
  mutate(t_warped = warp_time(t, peak_times[curve_id[1]], target_peak)) |>
  ungroup()

# 位置合わせ後の曲線を共通グリッドに再補間してから平均する
realigned_grid <- curves_aligned |>
  group_by(curve_id) |>
  group_modify(~ tibble(t = t_grid, f0 = approx(.x$t_warped, .x$f0, xout = t_grid, rule = 2)$y)) |>
  ungroup()

mean_aligned <- realigned_grid |> group_by(t) |> summarise(f0 = mean(f0), .groups = "drop")

p1 <- ggplot() +
  geom_line(data = curves, aes(t, f0, group = curve_id), colour = "grey70", linewidth = 0.3) +
  geom_line(data = mean_unaligned, aes(t, f0), colour = "#A8681F", linewidth = 1.1) +
  labs(title = "位置合わせ前(時間軸をそのまま重ねる)", x = "正規化時間", y = "F0(模式値)") +
  theme(plot.title = element_text(size = 10))

p2 <- ggplot() +
  geom_line(data = realigned_grid, aes(t, f0, group = curve_id), colour = "grey70", linewidth = 0.3) +
  geom_line(data = mean_aligned, aes(t, f0), colour = "#34518A", linewidth = 1.1) +
  labs(title = "位置合わせ後(各曲線のピーク時刻を揃えてから重ねる)", x = "正規化時間", y = "F0(模式値)") +
  theme(plot.title = element_text(size = 10))

library(patchwork)
p <- p1 + p2
ggsave(here::here("assets", "figures", "ch15", "fig15-03-fda-registration.png"),
       p, width = 8, height = 3.6, dpi = 320)
ggsave(here::here("assets", "figures", "ch15", "fig15-03-fda-registration.pdf"),
       p, width = 8, height = 3.6, device = cairo_pdf)

cat("個々の曲線の真のピーク高: 130\n")
cat("位置合わせ前の平均曲線のピーク高:", round(max(mean_unaligned$f0), 1), "\n")
cat("位置合わせ後の平均曲線のピーク高:", round(max(mean_aligned$f0), 1), "\n")
cat("完了。図はassets/figures/ch15/に保存した。\n")
